<p> <center> <a href="../../start_here_ja.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="../06_challenge_ja.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="../01_inference_endpoint_ja.ipynb">1</a>
        <a href="../02_introduction_mcp_ja.ipynb">2</a>
        <a href="../03_low_level_mcp_ja.ipynb">3</a>
        <a href="../04_langraph_agent_ja.ipynb">4</a>
        <a href="../05_nemo_agent_toolkit_ja.ipynb">5</a>
        <a href="../06_challenge_ja.ipynb">6</a>
        <a >7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"></span>
</div>

# OpenClaw

OpenClaw は、質問に答えるだけでなく、実際にアクションを実行することを目指して設計された personal AI アシスタントです — 事実上ほぼすべてのプラットフォームでユーザーの代わりに行動できます。

中核となるのは、自分のデバイス上で動かすセルフホスト型のアシスタントです。WhatsApp、Telegram、Slack、Discord、iMessage、Microsoft Teams、Signal など、すでに使っているコミュニケーションチャネルと統合できるので、どこにいても OpenClaw とやりとりできます。

一般的な AI チャットボットと OpenClaw が異なるのは、常時稼働の能動的な性質です。persistent memory、ペルソナのオンボーディング、コミュニケーション統合、バックグラウンドの heartbeat タスクをサポートしているため、cron ジョブの実行、リマインダーの設定、バックグラウンドでの自律動作が可能です。

OpenClaw はユーザーオーナーシップを重視して構築されています。あなたの context と skill は閉じたエコシステムではなく、あなた自身のコンピューター上で動作します。オープンソースで、新しい skill と機能を継続的に構築する成長中のコミュニティがあります。

テストを自律的に実行して pull request を開く coding コンパニオン、カレンダーや健康データに接続された個人ライフアシスタント、就寝中に email やタスクを処理する能動的な agent — OpenClaw は次世代の personal AI のためのオープンで拡張可能な基盤として位置づけられています。

**この lab では、Nemo Agent Toolkit ベースの workflow を agent skill を通じて OpenClaw agent に統合する手順を解説します。**

**このノートブックの残りでは、新しいターミナルを開いてそこでコマンドを実行してください。**

**OpenClaw はターミナルとファイルシステムへの完全で無制限なアクセスを持つため、隔離された環境(例:[brev.dev](https://brev.nvidia.com/) 上の VM)で実行することを推奨します。**

## セットアップ

以下の手順は `OpenClaw 2026.4.15` を `Ubuntu 22.04.5 LTS (Jammy Jellyfish)` 上でテストしました。
最新の手順は [公式ドキュメント](https://docs.openclaw.ai/start/getting-started) を参照してください。

このセットアップでは、OpenClaw は完全に Docker コンテナ内で動作します。
コンテナは OpenClaw リポジトリのセットアップスクリプトによって起動され、チャレンジリポジトリは volume として bind-mount されるので、完成させたソリューションファイルは手動でコピーすることなくコンテナ内からアクセスできます。
OpenClaw の設定と state はすべてコンテナのホームディレクトリ内に存在し、ホストシステムをクリーンに保ちます。

#### Docker コンテナを起動する

1. OpenClaw リポジトリを clone する

    ```
    git clone https://github.com/openclaw/openclaw.git
    ```

2. チャレンジリポジトリを clone する

    ```
    cd openclaw
    git clone https://github.com/openhackathons-org/agentic-ai-bootcamp.git
    ```

3. 以下のソースファイルを完成させたソリューションで更新する

    `agentic-ai-bootcamp/challenge/mcp-servers/invoice/src/server_http.py` — invoice データベース(顧客検索、invoice 行クエリ)を Model Context Protocol 経由で呼び出し可能な tool として公開する MCP server。

    `agentic-ai-bootcamp/challenge/nemo_agent_toolkit/workflow.yaml` — LLM をこれらの MCP tool に接続し、会話フローを処理する Nemo Agent Toolkit パイプライン定義。

    今コピーしておくことで、agent が music-store skill を呼び出した際に、完成させたソリューションをコンテナが使用するようになります。

4. OpenClaw docker コンテナ内のユーザーに所有権を更新する

    ```
    sudo chown -R 1000:1000 agentic-ai-bootcamp
    ```

5. Docker image を pull してセットアップスクリプトを実行する

    ```
    export OPENCLAW_IMAGE="ghcr.io/openclaw/openclaw:2026.4.15"
    export OPENCLAW_EXTRA_MOUNTS="./agentic-ai-bootcamp:/app/agentic-ai-bootcamp:rw"
    ./scripts/docker/setup.sh
    ```

    `I understand this is personal-by-default and shared/multi-user use requires lock-down.` と聞かれたら `yes` を選択。

    `Setup mode` では `quickstart` を選択。

    `Config handling` では `Use existing values` を選択。

    `Model/auth provider` では `Skip for now` を選択。

    `Filter models by provider` では `All providers` を選択。

    `Default model` では `Keep current` を選択。

    `Select channel` と聞かれたら `Skip for now` を選択。

    `Search provider` では `Skip for now` を選択。

    `Configure skills` と聞かれたら `No` を選択。

    API キーを有効化するか聞かれたら `No` を選択。

    `Enable hooks` と聞かれたら `Skip for now` を選択。

6. 出力された web UI(ダッシュボード)の URL を控えておく

    出力例:

    ```
    Control UI ─────────────────────────────────────────────────────────────────────────────────────────╮
    │                                                                                                   │
    │  Web UI: http://172.18.0.2:18789/                                                                 │
    │  Web UI (with token):                                                                             │
    │  http://172.18.0.2:18789/#token=<generated_token>                                                 │
    │  Gateway WS: ws://172.18.0.2:18789                                                                │
    │                                                                                                   │
    ├───────────────────────────────────────────────────────────────────────────────────────────────────╯
    └  Onboarding complete. Use the dashboard link above to control OpenClaw.
    ```

## OpenClaw を設定する

OpenClaw は openclaw CLI を使うか、config ファイルを直接編集するかのいずれかで設定できます。ここでは両方のアプローチを使用します。

#### music store agent を作成する [OpenClaw CLI]

```
docker compose run --rm openclaw-cli agents add music-store-agent
```

Workspace ディレクトリを聞かれたら `ENTER` を押す

agent のモデル/auth を設定するか聞かれたら `No` を選択

chat チャネルを設定するか聞かれたら `No` を選択

想定される出力:

```
Updated ~/.openclaw/openclaw.json
Workspace OK: ~/.openclaw/workspace/music-store-agent
Sessions OK: ~/.openclaw/agents/music-store-agent/sessions
```

#### OpenClaw ワークスペースの更新 [ファイルシステムから直接編集]

まず OpenClaw ワークスペースをホストシステム上で編集可能にする

```
chmod 777 -R $HOME/.openclaw
```

1. モデル属性を指定する

    OpenClaw の LLM バックエンドとして [nemotron 3 super 120b](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b) モデルを使用します。
    このモデルは最大 1M トークンをサポートしていますが、context window は 100k トークンに制限しています。

    以下の json で `<your_api_key>` を自分の API キーに置き換え、
    `$HOME/.openclaw/openclaw.json` に書き込みます。

    ```json
    "models": {
        "mode": "merge",
        "providers": {
            "nvidia-nim": {
                "baseUrl": "https://integrate.api.nvidia.com/v1",
                "apiKey": "<your_api_key>",
                "api": "openai-completions",
                "models": [
                    {
                        "id": "nvidia/nemotron-3-super-120b-a12b",
                        "name": "nvidia/nemotron-3-super-120b-a12b (Custom Provider)",
                        "reasoning": false,
                        "input": [
                            "text"
                        ],
                        "cost": {
                        "input": 0,
                        "output": 0,
                        "cacheRead": 0,
                        "cacheWrite": 0
                        },
                        "contextWindow": 100000,
                        "maxTokens": 4096
                    }
                ]
            }
        }
    }
    ```

2. agent 属性を指定する

    `$HOME/.openclaw/openclaw.json` の `music-store-agent` に `model` と `default` のキーを追加する。

    ```json
    "agents": {
        ...
        "list": [
          ...
          {
            "id": "music-store-agent",
            "name": "music-store-agent",
            ...
            "model": "nvidia/nemotron-3-super-120b-a12b",
            "default": true
          }
        ]
    }
    ```

3. music store agent を定義する

    `SOUL.md` で agent の振る舞いを指定する。

    ```
    cat > $HOME/.openclaw/workspace/music-store-agent/SOUL.md << 'EOF'
    # Music Store Agent

    You are a music store agent. Do not invent anything. All data comes from skills.

    ## Allowed Skills
    - music-store-assistant
    EOF
    ```

4. agent skill を作成する

    music store agent の中に skills フォルダーを作成する。

    ```
    mkdir -p $HOME/.openclaw/workspace/music-store-agent/skills/music-store-assistant
    ```

    `SKILL.md` に内容を追加する。

    この指示は agent に対し、NAT webserver の endpoint を叩いて workflow を呼び出す `curl` コマンドを発行するよう指示している。

    ```
    cat > $HOME/.openclaw/workspace/music-store-agent/skills/music-store-assistant/SKILL.md << 'EOF'
    ---
    name: music-store-assistant
    description: lookup information on tracks,artists,abums and process refunds.
    ---

    ### Query the music store assistant

    Replace <query> with question from user/agent. Do not remove any information.

    ```bash
    curl -X POST http://localhost:8001/generate \
    -H "Content-Type: application/json" \
    -d $'{
        "input_message": <query>
    }'
    EOF
    ```

## Workflow endpoint

以下のパート 1〜4 を実行する前に、docker コンテナにシェルで入ります。

```
docker exec -it <container_name> bash
```

agent skill は `http://localhost:8001/generate` を呼び出します。このリクエストを処理するためにコンテナ内で 2 つのバックグラウンドサービスが動いている必要があります:
- **MCP invoice server** (`mcp-server-invoice`) — invoice データベースを Model Context Protocol 経由で tool として公開する。NAT workflow はこれらの tool を呼び出して顧客と invoice 行を検索する。
- **NAT web server** — Nemo Agent Toolkit パイプラインをポート 8001 で実行する。自然言語クエリを受け取り、LLM を呼び出し、必要に応じて MCP tool を呼び出し、最終回答を返す。

以下のステップ 1〜4 でこれらのサービスを起動します。ステップ 5〜8 では OpenClaw を使って agent にクエリを投げ、エンドツーエンドで接続されていることを確認します。

1. uv をインストールする

    ```
    curl -LsSf https://astral.sh/uv/install.sh | sh
    source $HOME/.local/bin/env
    ```

2. mcp invoice server を起動する。

    ```
    uv run --directory agentic-ai-bootcamp/challenge/mcp-servers/invoice mcp-server-invoice &
    ```

3. nat web server を起動する

    ```
    cd agentic-ai-bootcamp/challenge/nemo_agent_toolkit
    uv sync
    source .venv/bin/activate
    nat serve --port 8001 --config_file workflow.yaml &
    ```

4. web endpoint をテストする

    ```
    curl -X POST http://localhost:8001/generate \
    -H "Content-Type: application/json" \
    -d $'{
        "input_message": "my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin"
    }'
    ```

    想定される出力:

    ```json
    {"value":"The invoice line IDs for tracks by Led Zeppelin are **267** and **268**."}
    ```

5. OpenClaw TUI を実行する

    ```
    openclaw tui
    ```

6. music store agent にクエリを投げる

    ```
    my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin
    ```

    想定される出力:

    ```
    The invoice line IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**.
    ```

7. セッション trace を確認する

    agent の sessions フォルダーのファイル一覧を表示する

    ```
    ls $HOME/.openclaw/agents/music-store-agent/sessions/
    ```

    想定される出力:

    ```
    38cadd64-fd9b-4858-93e5-f3fe45ea71d3.jsonl  sessions.json
    ```

    `<sessionId>` をディレクトリ内で見つけた jsonl ファイル名に置き換える

    ```
    cat $HOME/.openclaw/agents/music-store-agent/sessions/<sessionId>.jsonl
    ```

    想定される出力:

    ```json
    {"role":"user","content":[{"type":"text","text":"Sender (untrusted metadata):\n```json\n{\n  \"label\": \"openclaw-tui\",\n  \"id\": \"openclaw-tui\",\n  \"name\": \"openclaw-tui\",\n  \"username\": \"openclaw-tui\"\n}\n```\n\n[Fri 2026-04-10 07:39 UTC] my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin"}],"timestamp":1775806742065}}

    {"role":"assistant","content":[{"type":"thinking","thinking":"We need to get the invoice line IDs related to artist Led Zeppelin. First, I should check the available skills and see what tools I have. Looking at the skills, there's a music-store-assistant skill. I should read that to understand how to use it. Let me read the SKILL.md for music-store-assistant.\n\n","thinkingSignature":"reasoning_content"},{"type":"toolCall","id":"chatcmpl-tool-983ae2ee723dbbd6","name":"read","arguments":{"path":"/home/ubuntu/.openclaw/workspace-music-store-agent/skills/music-store-assistant/SKILL.md"}}],"api":"openai-completions","provider":"nvidia-nim","timestamp":1775806748147}

    {"role":"toolResult","toolCallId":"chatcmpl-tool-983ae2ee723dbbd6","toolName":"read","content":[{"type":"text","text":"---\nname: music-store-assistant\ndescription: lookup information on tracks,artists,abums and process refunds.\n---\n\n### Query the music store assistant\n\nReplace <query> with question from user/agent. Do not remove any information.\n\n```bash\ncurl -X POST http://localhost:8001/generate \\\n-H \"Content-Type: application/json\" \\\n-d $'{\n    \"input_message\": <query>\n}'\n"}],"timestamp":1775806748162}

    {"role":"assistant","content":[{"type":"thinking","thinking":"We need to call the music store assistant skill to get the invoice line IDs. Let me execute the curl command with the user's query.\n\n","thinkingSignature":"reasoning_content"},{"type":"toolCall","id":"chatcmpl-tool-bdef75da29fa3b69","name":"exec","arguments":{"command":"curl -X POST http://localhost:8001/generate -H 'Content-Type: application/json' -d $'{\"input_message\": \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin\"}'"}}],"api":"openai-completions","provider":"nvidia-nim","timestamp":1775806752331}

    {"role":"toolResult","toolCallId":"chatcmpl-tool-bdef75da29fa3b69","toolName":"exec","content":[{"type":"text","text":"{\"detail\":\"Invalid JSON: Expecting value: line 1 column 1 (char 0)\"}"}],"timestamp":1775806752461}

    {"role":"assistant","content":[{"type":"thinking","thinking":"The user is asking for invoice line IDs related to the artist Led Zeppelin. The previous attempt failed because of a JSON parsing issue with the bash quoting syntax. Let me retry with proper JSON formatting using standard double quotes.\n\n","thinkingSignature":"reasoning_content"},{"type":"toolCall","id":"chatcmpl-tool-8fb781cff172d362","name":"exec","arguments":{"command":"curl -X POST http://localhost:8001/generate -H \"Content-Type: application/json\" -d '{\"input_message\": \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin\"}'"}}],"api":"openai-completions","provider":"nvidia-nim","timestamp":1775806755872}

    {"role":"toolResult","toolCallId":"chatcmpl-tool-8fb781cff172d362","toolName":"exec","content":[{"type":"text","text":"{\"value\":\"The invoice line IDs for tracks by Led Zeppelin are **267** and **268**.\"}"}],"timestamp":1775806756134}

    {"role":"assistant","content":[{"type":"thinking","thinking":"We have received the response from the music store assistant. The invoice line IDs for tracks by Led Zeppelin for Aaron Mitchell are 267 and 268. Let me respond to the user clearly.\n\n","thinkingSignature":"reasoning_content"},{"type":"text","text":"The invoice line IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**."}],"api":"openai-completions","provider":"nvidia-nim","timestamp":1775806758450}
    ```

    インタラクションのサマリーを以下に示します。

    1. ユーザーリクエスト
    Aaron Mitchell(電話番号:+1 (204) 452-6452)が、Led Zeppelin に関連する invoice 行 ID を尋ねるメッセージを送信した。
    2. アシスタントの思考
    アシスタントは内部で推論し、適切な tool を見つける必要があると判断し、music-store-assistant skill を特定して、その使い方を理解するために skill ファイルを読むことにした。
    3. アシスタントが skill ファイルを読む
    アシスタントは read tool を呼び出して `SKILL.md` を読み、music-store-assistant の使い方を理解した。
    4. skill 指示の取得
    SKILL.md には、クエリを `http://localhost:8001/generate` への curl POST リクエストで送信すべきと書かれていた。
    5. 1 回目の API 呼び出し(失敗)
    アシスタントは curl リクエストを発行したが、`$'...'` という bash の quoting syntax を使ったため JSON パースエラーが発生し、API がエラーを返した。
    6. 2 回目の API 呼び出し(成功)
    アシスタントは quoting を修正し、標準的なダブルクォートを使って同じリクエストを再送信した。今度は server がリクエストを正常に処理した。
    7. API レスポンス受信
    music store API は結果を返した:invoice 行 ID 267 と 268 が Aaron Mitchell の Led Zeppelin 購入に紐づいている。
    8. ユーザーへの最終回答
    アシスタントは結果を明確に伝え、Aaron Mitchell の Led Zeppelin に関する invoice 行 ID が 267 と 268 であると伝えた。

8. 最初に控えておいたダッシュボード URL にアクセスする

    左側のペインで `Chat` をクリックします。以下のような画面が見えるはずです。

    ![openclaw dashboard](../images/openclaw-dashboard.png)

    ダッシュボードの他の項目も自由に探索してください。上記で行った設定変更のいくつかが見えるはずです。

# NemoClaw

NemoClaw は NVIDIA が提供するオープンソーススタックで、自律 AI agent(特に OpenClaw のような常時稼働アシスタント)を運用する際のセキュリティ、プライバシー、信頼性のギャップを解決することを目指しています。
NemoClaw が取り組む問題の概要を以下の表に示します。

| 観点                  | OpenClaw のベースライン                                                                                                         | OpenClaw 上の NemoClaw                                                                                                    |
| ----------------------- | ------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------ |
| システムアクセス           | デフォルトで shell、ファイル、スクリプトへの完全アクセス。広範な権限で動作することが多い                         | サンドボックス化されたランタイム内で動作し、ポリシーで制御された限定的な機能のみを持つ                              |
| ネットワーク露出        | 弱い認証や安全でないデフォルトでインターネットに直接公開されているインスタンスが多い                               | ポリシー経由で出入りのネットワークトラフィックを制御した堅牢なデプロイ向けに設計されている                         |
| Skill エコシステム         | 検証されていないコミュニティ skill が多数あり、マルウェアや有害な指示も報告されている                      | skill 自体を「直す」のではなく、サンドボックスと skill ごとのポリシーで skill ができることを制限する                    |
| プロンプト注入の影響 | 任意のコード実行、認証情報漏洩、データ持ち出しにつながり得る。組み込みの緩和策はほとんどない | プライバシールーターと guardrail で、悪意あるプロンプトが引き起こす機密データフローを検出・ブロック可能          |
| 機密情報とデータ処理 | 平文での鍵漏洩、ログ露出、不明確なデータフロー制御の事例がある                            | サンドボックスの外に出るデータに対する一元的なポリシー。機密アクセスの試行をモニタリング         |
| エンタープライズ制御     | 監査、最小権限、コンプライアンス制御のネイティブサポートは限定的                                    | 監査証跡、構成可能なセキュリティ/プライバシーポリシー、エンタープライズセキュリティツールへの統合を追加 |

NemoClaw の動作についての詳細は [こちら](https://docs.nvidia.com/nemoclaw/latest/about/how-it-works.html) を参照してください。

<p> <center> <a href="../../start_here_ja.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="../06_challenge_ja.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="../01_inference_endpoint_ja.ipynb">1</a>
        <a href="../02_introduction_mcp_ja.ipynb">2</a>
        <a href="../03_low_level_mcp_ja.ipynb">3</a>
        <a href="../04_langraph_agent_ja.ipynb">4</a>
        <a href="../05_nemo_agent_toolkit_ja.ipynb">5</a>
        <a href="../06_challenge_ja.ipynb">6</a>
        <a >7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"></span>
</div>